In [5]:
import subprocess
import os
import sys

def check_java_installation():
    """Check if Java is installed and find its path"""
    try:
        # Try to get Java version
        result = subprocess.run(['java', '-version'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✓ Java is installed")
            print("Java version info:")
            print(result.stderr)
            return True
        else:
            print("✗ Java command failed")
            return False
    except FileNotFoundError:
        print("✗ Java is not installed or not in PATH")
        return False

check_java_installation()

✗ Java is not installed or not in PATH


False

In [6]:
def find_java_home():
    """Try to find Java installation path"""
    possible_paths = []
    
    # Common Java installation paths
    common_paths = [
        # Linux paths
        "/usr/lib/jvm/java-11-openjdk-amd64",
        "/usr/lib/jvm/java-8-openjdk-amd64", 
        "/usr/lib/jvm/java-11-openjdk",
        "/usr/lib/jvm/java-8-openjdk",
        # macOS paths
        "/Library/Java/JavaVirtualMachines/jdk-11.jdk/Contents/Home",
        "/Library/Java/JavaVirtualMachines/jdk-8.jdk/Contents/Home",
        "/usr/local/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home",
        # Windows paths (if running on Windows)
        "C:/Program Files/Java/jdk-11",
        "C:/Program Files/Java/jdk-8",
        "C:/Program Files/Java/jre1.8.0_291",
    ]
    
    for path in common_paths:
        if os.path.exists(path):
            possible_paths.append(path)
            print(f"✓ Found Java at: {path}")
    
    return possible_paths

java_paths = find_java_home()

In [8]:
def setup_java_home():
    """Set JAVA_HOME environment variable"""
    
    # Try to auto-detect or use common path
    java_paths = find_java_home()
    
    if java_paths:
        # Use the first found Java path
        java_home = java_paths[0]
        os.environ['JAVA_HOME'] = java_home
        print(f"✓ Set JAVA_HOME to: {java_home}")
        return True
    else:
        print("✗ Could not auto-detect Java installation")
        print("\nPlease install Java or set JAVA_HOME manually:")
        print("- Install Java 8 or 11 from: https://adoptium.net/")
        print("- Or set JAVA_HOME manually: os.environ['JAVA_HOME'] = '/your/java/path'")
        return False

if setup_java_home():
    print(f"JAVA_HOME is now: {os.environ['JAVA_HOME']}")

✗ Could not auto-detect Java installation

Please install Java or set JAVA_HOME manually:
- Install Java 8 or 11 from: https://adoptium.net/
- Or set JAVA_HOME manually: os.environ['JAVA_HOME'] = '/your/java/path'


In [9]:
def install_java_instructions():
    """Provide instructions for installing Java"""
    print("\n" + "="*60)
    print("JAVA INSTALLATION INSTRUCTIONS")
    print("="*60)
    
    print("\n1. Install Java 8 or 11 from:")
    print("   https://adoptium.net/")
    print("   OR")
    print("   https://www.oracle.com/java/technologies/downloads/")
    
    print("\n2. For Ubuntu/Debian:")
    print("   sudo apt update")
    print("   sudo apt install openjdk-11-jdk")
    
    print("\n3. For macOS with Homebrew:")
    print("   brew install openjdk@11")
    
    print("\n4. After installation, find Java path and set:")
    print("   import os")
    print("   os.environ['JAVA_HOME'] = '/path/to/your/java'")
    print("="*60)

# If no Java found, show installation instructions
if not java_paths:
    install_java_instructions()


JAVA INSTALLATION INSTRUCTIONS

1. Install Java 8 or 11 from:
   https://adoptium.net/
   OR
   https://www.oracle.com/java/technologies/downloads/

2. For Ubuntu/Debian:
   sudo apt update
   sudo apt install openjdk-11-jdk

3. For macOS with Homebrew:
   brew install openjdk@11

4. After installation, find Java path and set:
   import os
   os.environ['JAVA_HOME'] = '/path/to/your/java'


In [11]:
# Try using findspark which can help auto-configure Spark
try:
    import findspark
    findspark.init()
    print("✓ findspark initialized")
except ImportError:
    print("Installing findspark...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "findspark"])
    import findspark
    findspark.init()
    print("✓ findspark installed and initialized")

Installing findspark...
✓ findspark installed and initialized


In [13]:
def create_spark_session():
    """Create Spark session with proper configuration"""
    try:
        from pyspark.sql import SparkSession
        
        spark = SparkSession.builder \
            .appName("InternationalDebtAnalysis") \
            .config("spark.sql.adaptive.enabled", "true") \
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
            .config("spark.executor.memory", "1g") \
            .config("spark.driver.memory", "1g") \
            .config("spark.driver.bindAddress", "127.0.0.1") \
            .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
            .master("local[*]") \
            .getOrCreate()
        
        print("✓ Spark session created successfully!")
        
        # Test with simple operation
        test_df = spark.range(5)
        print(f"✓ Test DataFrame count: {test_df.count()}")
        
        return spark
        
    except Exception as e:
        print(f"✗ Error creating Spark session: {e}")
        return None

# Try to create Spark session
spark = create_spark_session()

✗ Error creating Spark session: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.


JAVA_HOME is not set


In [14]:
if spark is None:
    print("\n" + "="*60)
    print("USING PANDAS AS FALLBACK SOLUTION")
    print("="*60)
    
    try:
        import pandas as pd
        import numpy as np
        
        print("✓ Pandas imported successfully")
        print("We'll use pandas for data analysis instead of PySpark")
        
        # Load data with pandas
        try:
            df_pandas = pd.read_csv("international_debt_with_missing_values.csv")
            print("✓ Dataset loaded with pandas")
            print(f"Dataset shape: {df_pandas.shape}")
            print(f"Columns: {list(df_pandas.columns)}")
            print("\nFirst 5 rows:")
            print(df_pandas.head())
            
        except FileNotFoundError:
            print("Creating sample data for demonstration...")
            # Create sample data
            sample_data = {
                'country_name': ['United States', 'China', 'India', 'Germany', 'Japan'],
                'country_code': ['USA', 'CHN', 'IND', 'DEU', 'JPN'],
                'indicator_name': ['External debt stocks'] * 5,
                'indicator_code': ['DT.DOD.DECT.CD'] * 5,
                'year_2010': [1000000.0, 500000.0, 300000.0, 400000.0, 350000.0],
                'year_2011': [1100000.0, 550000.0, 320000.0, 420000.0, 360000.0],
                'year_2012': [1200000.0, 600000.0, 350000.0, 430000.0, 370000.0],
            }
            df_pandas = pd.DataFrame(sample_data)
            print("Sample data created:")
            print(df_pandas)
            
    except ImportError:
        print("✗ Pandas is not available either")
        print("Please install pandas: pip install pandas")


USING PANDAS AS FALLBACK SOLUTION
✓ Pandas imported successfully
We'll use pandas for data analysis instead of PySpark
✓ Dataset loaded with pandas
Dataset shape: (2357, 5)
Columns: ['country_name', 'country_code', 'indicator_name', 'indicator_code', 'debt']

First 5 rows:
  country_name country_code  \
0  Afghanistan          AFG   
1  Afghanistan          NaN   
2          NaN          AFG   
3  Afghanistan          AFG   
4  Afghanistan          AFG   

                                      indicator_name  indicator_code  \
0  Disbursements on external debt, long-term (DIS...  DT.DIS.DLXF.CD   
1  Interest payments on external debt, long-term ...  DT.INT.DLXF.CD   
2                  PPG, bilateral (AMT, current US$)  DT.AMT.BLAT.CD   
3                  PPG, bilateral (DIS, current US$)  DT.DIS.BLAT.CD   
4                  PPG, bilateral (INT, current US$)  DT.INT.BLAT.CD   

         debt  
0  72894453.7  
1  53239440.1  
2  61739336.9  
3  49114729.4  
4  39903620.1  


In [16]:
import os
# Common paths - try one of these:
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'  # Linux
# os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/jdk-11.jdk/Contents/Home'  # macOS
# os.environ['JAVA_HOME'] = 'C:/Program Files/Java/jdk-11'  # Windows

In [17]:
# Initialize Spark Session
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("InternationalDebtAnalysis") \
    .getOrCreate()

# Load the dataset
df = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("international_debt_with_missing_values.csv")

# Display schema and basic info
print("Dataset Schema:")
df.printSchema()

print("\nFirst 10 rows:")
df.show(10, truncate=False)

print(f"Total records: {df.count()}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/18 23:31:49 WARN Utils: Your hostname, DESKTOP-PEEMDC8, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/18 23:31:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/18 23:31:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Dataset Schema:
root
 |-- country_name: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- indicator_name: string (nullable = true)
 |-- indicator_code: string (nullable = true)
 |-- debt: double (nullable = true)


First 10 rows:
+------------+------------+----------------------------------------------------------------+--------------+-------------+
|country_name|country_code|indicator_name                                                  |indicator_code|debt         |
+------------+------------+----------------------------------------------------------------+--------------+-------------+
|Afghanistan |AFG         |Disbursements on external debt, long-term (DIS, current US$)    |DT.DIS.DLXF.CD|7.28944537E7 |
|Afghanistan |NULL        |Interest payments on external debt, long-term (INT, current US$)|DT.INT.DLXF.CD|5.32394401E7 |
|NULL        |AFG         |PPG, bilateral (AMT, current US$)                               |DT.AMT.BLAT.CD|6.17393369E7 |
|Afghanistan |

In [18]:
# 1. What is the total amount of debt owed by all countries in the dataset?
total_debt = df.select(sum("debt")).collect()[0][0]
print(f"1. Total amount of debt owed by all countries: ${total_debt:,.2f}")

1. Total amount of debt owed by all countries: $2,823,893,300,259.09


In [19]:
# 2. How many distinct countries are recorded in the dataset?
distinct_countries = df.select("country_name").distinct().count()
print(f"2. Number of distinct countries: {distinct_countries}")

# Show some country names
print("\nSample of countries:")
df.select("country_name").distinct().show(20, truncate=False)

2. Number of distinct countries: 125

Sample of countries:
+--------------------------------------------+
|country_name                                |
+--------------------------------------------+
|South Asia                                  |
|Chad                                        |
|Paraguay                                    |
|Congo, Dem. Rep.                            |
|Senegal                                     |
|Cabo Verde                                  |
|Least developed countries: UN classification|
|Macedonia, FYR                              |
|Guyana                                      |
|Eritrea                                     |
|Philippines                                 |
|Djibouti                                    |
|Tonga                                       |
|Fiji                                        |
|Turkey                                      |
|Malawi                                      |
|Comoros                                     |
|

In [20]:
# 3. What are the distinct types of indicators and what do they represent?
print("3. Distinct debt indicators and their descriptions:")

# Group by indicator_name and count occurrences
indicators_summary = df.groupBy("indicator_name", "indicator_code") \
    .agg(count("*").alias("count")) \
    .orderBy(desc("count"))

indicators_summary.show(30, truncate=False)

# Get unique indicator codes and their meanings
print("\nIndicator codes and their likely meanings:")
indicator_codes = df.select("indicator_code").distinct().filter(col("indicator_code").isNotNull())
indicator_codes.show(30, truncate=False)

# Common indicator patterns
print("\nCommon indicator patterns:")
df.select("indicator_code").distinct() \
    .filter(col("indicator_code").isNotNull()) \
    .rdd.map(lambda x: x[0]) \
    .filter(lambda x: x is not None) \
    .take(20)

3. Distinct debt indicators and their descriptions:
+-------------------------------------------------------------------------------------+--------------+-----+
|indicator_name                                                                       |indicator_code|count|
+-------------------------------------------------------------------------------------+--------------+-----+
|PPG, official creditors (INT, current US$)                                           |DT.INT.OFFT.CD|107  |
|PPG, multilateral (AMT, current US$)                                                 |DT.AMT.MLAT.CD|104  |
|PPG, official creditors (AMT, current US$)                                           |DT.AMT.OFFT.CD|104  |
|Principal repayments on external debt, long-term (AMT, current US$)                  |DT.AMT.DLXF.CD|102  |
|Disbursements on external debt, long-term (DIS, current US$)                         |DT.DIS.DLXF.CD|100  |
|Interest payments on external debt, long-term (INT, current US$)           

['DT.AMT.DLXF.CD',
 'DT.DIS.PRVT.CD',
 'DT.INT.MLAT.CD',
 'DT.INT.PCBK.CD',
 'DT.AMT.OFFT.CD',
 'DT.AMT.PRVT.CD',
 'DT.INT.DPNG.CD',
 'DT.INT.PROP.CD',
 'DT.AMT.PROP.CD',
 'DT.INT.BLAT.CD',
 'DT.INT.PBND.CD',
 'DT.DIS.MLAT.CD',
 'DT.DIS.OFFT.CD',
 'DT.DIS.DLXF.CD',
 'DT.DIS.PCBK.CD',
 'DT.DIS.BLAT.CD',
 'DT.DIS.PROP.CD',
 'DT.INT.DLXF.CD',
 'DT.AMT.PCBK.CD',
 'DT.AMT.BLAT.CD']

In [21]:
# 4. Which country has the highest total debt and how much does it owe?
print("4. Countries with highest total debt:")

# Calculate total debt per country (handling null values)
country_total_debt = df.filter(col("debt").isNotNull()) \
    .groupBy("country_name") \
    .agg(sum("debt").alias("total_debt")) \
    .orderBy(desc("total_debt"))

country_total_debt.show(20, truncate=False)

# Top country
top_debt_country = country_total_debt.first()
print(f"\nCountry with highest total debt: {top_debt_country['country_name']}")
print(f"Total debt: ${top_debt_country['total_debt']:,.2f}")

4. Countries with highest total debt:
+--------------------------------------------+---------------------+
|country_name                                |total_debt           |
+--------------------------------------------+---------------------+
|NULL                                        |2.751793088518001E11 |
|China                                       |2.664557603373E11    |
|South Asia                                  |2.4368837359750003E11|
|Brazil                                      |1.761855840362E11    |
|Russian Federation                          |1.673876658084E11    |
|Least developed countries: UN classification|1.5225435490250003E11|
|Turkey                                      |1.366056037787E11    |
|IDA only                                    |1.2850532490320003E11|
|Mexico                                      |1.2334979806029999E11|
|India                                       |1.2011644123790001E11|
|Indonesia                                   |9.25100111052E10   

In [22]:
# 5. What is the average debt across different debt indicators?
print("5. Average debt by indicator type:")

# Group by indicator_name and calculate average debt
avg_debt_by_indicator = df.filter(col("debt").isNotNull()) \
    .groupBy("indicator_name") \
    .agg(
        avg("debt").alias("avg_debt"),
        count("*").alias("record_count")
    ) \
    .orderBy(desc("avg_debt"))

avg_debt_by_indicator.show(30, truncate=False)

# Also show by indicator_code pattern
print("\nAverage debt by indicator code pattern:")
# Extract main indicator category from code
df_with_category = df.withColumn(
    "indicator_category", 
    regexp_extract(col("indicator_code"), r"^DT\.([A-Z]+)\.", 1)
)

avg_debt_by_category = df_with_category.filter(col("debt").isNotNull()) \
    .groupBy("indicator_category") \
    .agg(
        avg("debt").alias("avg_debt"),
        count("*").alias("record_count")
    ) \
    .orderBy(desc("avg_debt"))

avg_debt_by_category.show(truncate=False)

5. Average debt by indicator type:
+-------------------------------------------------------------------------------------+--------------------+------------+
|indicator_name                                                                       |avg_debt            |record_count|
+-------------------------------------------------------------------------------------+--------------------+------------+
|Principal repayments on external debt, long-term (AMT, current US$)                  |6.3851028591592245E9|103         |
|Principal repayments on external debt, private nonguaranteed (PNG) (AMT, current US$)|5.617528432625396E9 |63          |
|Disbursements on external debt, long-term (DIS, current US$)                         |1.9525070879481814E9|110         |
|PPG, private creditors (AMT, current US$)                                            |1.813818527970731E9 |82          |
|NULL                                                                                 |1.5384840095781255E9|224

In [23]:
# 6. Which country has made the highest number of principal repayments?
print("6. Principal repayments analysis:")

# Filter for principal repayment indicators
principal_repayments = df.filter(
    (col("indicator_name").like("%Principal repayments%")) | 
    (col("indicator_code") == "DT.AMT.DLXF.CD")
)

print(f"Number of principal repayment records: {principal_repayments.count()}")

# Countries with highest principal repayments
principal_by_country = principal_repayments.filter(col("debt").isNotNull()) \
    .groupBy("country_name") \
    .agg(sum("debt").alias("total_principal_repayments")) \
    .orderBy(desc("total_principal_repayments"))

principal_by_country.show(20, truncate=False)

top_repayer = principal_by_country.first()
print(f"\nCountry with highest principal repayments: {top_repayer['country_name']}")
print(f"Total principal repayments: ${top_repayer['total_principal_repayments']:,.2f}")

6. Principal repayments analysis:
Number of principal repayment records: 195
+--------------------------------------------+--------------------------+
|country_name                                |total_principal_repayments|
+--------------------------------------------+--------------------------+
|China                                       |1.686116070495E11         |
|NULL                                        |1.1719142988350002E11     |
|Russian Federation                          |1.093899168084E11         |
|Turkey                                      |9.14712284358E10          |
|South Asia                                  |7.31271459606E10          |
|Kazakhstan                                  |5.38394539864E10          |
|Brazil                                      |4.18314440533E10          |
|India                                       |3.19235070008E10          |
|Mexico                                      |3.0557842117E10           |
|Least developed countries: UN clas

In [24]:
# 7. What is the most common debt indicator across all countries?
print("7. Most common debt indicators:")

common_indicators = df.groupBy("indicator_name", "indicator_code") \
    .agg(count("*").alias("frequency")) \
    .orderBy(desc("frequency"))

common_indicators.show(20, truncate=False)

most_common = common_indicators.first()
print(f"\nMost common debt indicator: {most_common['indicator_name']}")
print(f"Frequency: {most_common['frequency']} records")

7. Most common debt indicators:
+-------------------------------------------------------------------------------------+--------------+---------+
|indicator_name                                                                       |indicator_code|frequency|
+-------------------------------------------------------------------------------------+--------------+---------+
|PPG, official creditors (INT, current US$)                                           |DT.INT.OFFT.CD|107      |
|PPG, multilateral (AMT, current US$)                                                 |DT.AMT.MLAT.CD|104      |
|PPG, official creditors (AMT, current US$)                                           |DT.AMT.OFFT.CD|104      |
|Principal repayments on external debt, long-term (AMT, current US$)                  |DT.AMT.DLXF.CD|102      |
|Disbursements on external debt, long-term (DIS, current US$)                         |DT.DIS.DLXF.CD|100      |
|Interest payments on external debt, long-term (INT, current US$

In [25]:
# 8. Identify any other key debt trends and summarize your findings
print("8. Additional Debt Trends Analysis:")

# Trend 1: Debt distribution by country
print("\nTrend 1: Debt Distribution Statistics")
debt_stats = df.filter(col("debt").isNotNull()).select(
    mean("debt").alias("mean_debt"),
    stddev("debt").alias("std_dev"),
    min("debt").alias("min_debt"),
    max("debt").alias("max_debt"),
    count("debt").alias("non_null_records")
).collect()[0]

print(f"Average debt per record: ${debt_stats['mean_debt']:,.2f}")
print(f"Debt standard deviation: ${debt_stats['std_dev']:,.2f}")
print(f"Minimum debt: ${debt_stats['min_debt']:,.2f}")
print(f"Maximum debt: ${debt_stats['max_debt']:,.2f}")

# Trend 2: Missing data analysis
print("\nTrend 2: Data Quality Analysis")
total_records = df.count()
null_debt_records = df.filter(col("debt").isNull()).count()
null_country_records = df.filter(col("country_name").isNull()).count()
null_indicator_records = df.filter(col("indicator_name").isNull()).count()

print(f"Total records: {total_records}")
print(f"Records with null debt values: {null_debt_records} ({null_debt_records/total_records*100:.1f}%)")
print(f"Records with null country names: {null_country_records} ({null_country_records/total_records*100:.1f}%)")
print(f"Records with null indicator names: {null_indicator_records} ({null_indicator_records/total_records*100:.1f}%)")

# Trend 3: Debt by region/country groups
print("\nTrend 3: Regional/Group Analysis")
# Identify regional groupings or country classifications
regional_keywords = ["IDA", "least developed", "South Asia", "Africa"]

for keyword in regional_keywords:
    region_debt = df.filter(lower(col("country_name")).like(f"%{keyword.lower()}%")) \
        .filter(col("debt").isNotNull()) \
        .agg(sum("debt").alias("total_debt")).collect()[0]["total_debt"]
    if region_debt:
        print(f"Total debt for '{keyword}': ${region_debt:,.2f}")

# Trend 4: Debt type analysis
print("\nTrend 4: Debt Type Breakdown")
debt_types = {
    "Bilateral": "BLAT",
    "Multilateral": "MLAT", 
    "Bonds": "PBND",
    "Commercial Banks": "PCBK",
    "Official Creditors": "OFFT",
    "Private Creditors": "PRVT"
}

for debt_type, code in debt_types.items():
    type_debt = df.filter(col("indicator_code").like(f"%{code}%")) \
        .filter(col("debt").isNotNull()) \
        .agg(sum("debt").alias("total_debt")).collect()[0]["total_debt"]
    if type_debt:
        print(f"Total {debt_type} debt: ${type_debt:,.2f}")

8. Additional Debt Trends Analysis:

Trend 1: Debt Distribution Statistics
Average debt per record: $1,341,517,007.25
Debt standard deviation: $5,404,482,579.08
Minimum debt: $0.00
Maximum debt: $96,218,620,835.70

Trend 2: Data Quality Analysis
Total records: 2357
Records with null debt values: 252 (10.7%)
Records with null country names: 234 (9.9%)
Records with null indicator names: 250 (10.6%)

Trend 3: Regional/Group Analysis


Total debt for 'IDA': $128,505,324,903.20
Total debt for 'least developed': $152,254,354,902.50
Total debt for 'South Asia': $243,688,373,597.50
Total debt for 'Africa': $34,155,183,625.90

Trend 4: Debt Type Breakdown
Total Bilateral debt: $202,178,807,287.80
Total Multilateral debt: $133,552,053,814.00
Total Bonds debt: $120,780,608,866.30
Total Commercial Banks debt: $64,338,985,180.30
Total Official Creditors debt: $318,260,388,512.70
Total Private Creditors debt: $233,701,584,869.70


In [26]:
# Summary of Findings
print("=" * 80)
print("SUMMARY OF KEY FINDINGS")
print("=" * 80)

print(f"1. TOTAL DEBT: ${total_debt:,.2f} owed by all countries combined")
print(f"2. COUNTRY DIVERSITY: {distinct_countries} distinct countries in dataset")
print(f"3. DEBT CONCENTRATION: Top country owes ${top_debt_country['total_debt']:,.2f}")
print(f"4. PRINCIPAL REPAYMENTS: Highest repayer paid ${top_repayer['total_principal_repayments']:,.2f}")
print(f"5. DATA QUALITY: {null_debt_records/total_records*100:.1f}% records missing debt values")
print(f"6. COMMON INDICATOR: '{most_common['indicator_name']}' appears {most_common['frequency']} times")

print("\nKEY TRENDS:")
print("- Debt is heavily concentrated in a few major economies")
print("- Principal repayments and disbursements are major debt activities")
print("- Bilateral and multilateral official creditors play significant roles")
print("- Some countries show very high debt levels requiring further investigation")
print("- Data quality issues exist, particularly with missing country names and debt values")

SUMMARY OF KEY FINDINGS
1. TOTAL DEBT: $2,823,893,300,259.09 owed by all countries combined
2. COUNTRY DIVERSITY: 125 distinct countries in dataset
3. DEBT CONCENTRATION: Top country owes $275,179,308,851.80
4. PRINCIPAL REPAYMENTS: Highest repayer paid $168,611,607,049.50
5. DATA QUALITY: 10.7% records missing debt values
6. COMMON INDICATOR: 'PPG, official creditors (INT, current US$)' appears 107 times

KEY TRENDS:
- Debt is heavily concentrated in a few major economies
- Principal repayments and disbursements are major debt activities
- Bilateral and multilateral official creditors play significant roles
- Some countries show very high debt levels requiring further investigation
- Data quality issues exist, particularly with missing country names and debt values


In [ ]:
# Stop Spark session
spark.stop()